# PubChem collection from DSLD: inspect before saving
Collect candidate chemical records for ingredients and forms occurring in the 400 DSLD labels from the ten supplement searches. This includes co-ingredients, not just the ten category names. Search-category membership is provenance, not verified ingredient identity. Excipients (`otheringredients`) are excluded.

All new responses remain in memory until the final save cell is explicitly enabled. Review coverage, failures, candidate counts and serialized size first. No RxNorm or MeSH collection is performed here. Raw synonym lists and all candidate CIDs are retained without asserting equivalence.

API references: https://pubchem.ncbi.nlm.nih.gov/docs/pug-rest and https://pubchem.ncbi.nlm.nih.gov/docs/dynamic-request-throttling


In [1]:
import json
import time
import hashlib
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote
import requests

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DSLD_DIR = PROJECT_ROOT / 'data/raw/dsld'
RAW_DIR = PROJECT_ROOT / 'data/raw/pubchem'
BASE_URL = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug'
SUPPLEMENTS = ['Magnesium', 'Vitamin D', 'Ashwagandha', 'Melatonin', 'Omega-3',
               'Vitamin B12', 'Iron', 'Turmeric', 'Probiotics', 'Creatine']
MIN_INTERVAL = 0.35
MAX_ATTEMPTS = 3
BATCH_SIZE = 25
SAVE_DATA = False  # Review preview_report before enabling the final cell.
session = requests.Session()
session.headers['User-Agent'] = 'SupplementRx-AI-Academic-POC/1.0'
def utcnow():
    return datetime.now(timezone.utc).isoformat()


## 1. Build the query inventory with DSLD provenance
Recursively visit ingredient rows and forms. Case/whitespace grouping only reduces duplicate requests; retain all exact original names and occurrences. Ingredient IDs and JSON paths allow later review without changing raw DSLD data.


In [2]:
def walk_rows(rows, path='ingredientRows'):
    for i, row in enumerate(rows or []):
        row_path = f'{path}[{i}]'
        yield row, row_path
        yield from walk_rows(row.get('nestedRows'), row_path + '.nestedRows')

def build_inventory():
    manifest_bytes = (DSLD_DIR / 'collection_manifest.json').read_bytes()
    manifest = json.loads(manifest_bytes)
    groups = {}
    label_hashes = {}
    for label_id, categories in manifest['category_matches'].items():
        categories = sorted(set(categories) & set(SUPPLEMENTS))
        if not categories:
            continue
        path = DSLD_DIR / 'labels' / f'label_{label_id}.json'
        raw = path.read_bytes()  # Missing/invalid source records fail explicitly.
        label_hashes[label_id] = hashlib.sha256(raw).hexdigest()
        label = json.loads(raw)
        if str(label['id']) != label_id:
            raise ValueError(f'Label ID mismatch: {path}')
        for row, row_path in walk_rows(label.get('ingredientRows')):
            objects = [(row, row_path, 'ingredient')]
            objects += [(form, f'{row_path}.forms[{i}]', 'form')
                        for i, form in enumerate(row.get('forms') or [])]
            for obj, obj_path, kind in objects:
                original = obj.get('name')
                if not original or not original.strip():
                    continue
                key = ' '.join(original.casefold().split())
                group = groups.setdefault(key, {'query': original.strip(), 'variants': [],
                                               'occurrences': []})
                if original not in group['variants']:
                    group['variants'].append(original)
                group['occurrences'].append({
                    'dsld_id': label_id, 'search_categories': categories,
                    'ingredient_id': obj.get('ingredientId'), 'name': original,
                    'kind': kind, 'json_path': obj_path})
    provenance = {'manifest_sha256': hashlib.sha256(manifest_bytes).hexdigest(),
                  'label_sha256': label_hashes}
    return dict(sorted(groups.items())), provenance

query_inventory, dsld_provenance = build_inventory()
print('Source labels:', len(dsld_provenance['label_sha256']))
print('Distinct original names:', len({v for g in query_inventory.values() for v in g['variants']}))
print('Query groups:', len(query_inventory))


Source labels: 400
Distinct original names: 753
Query groups: 739


## 2. Rate-limited HTTP requests with provenance and bounded retries
Only an explicit PubChem `PUGREST.NotFound` 404 counts as no match. Network failures, malformed responses and other errors remain errors. Request timing, response status, URL, raw JSON and throttle headers are retained. No response is written to disk here.


In [3]:
_last_request = 0.0

def request_json(url):
    global _last_request
    history = []
    for attempt in range(1, MAX_ATTEMPTS + 1):
        time.sleep(max(0, MIN_INTERVAL - (time.monotonic() - _last_request)))
        _last_request = time.monotonic()
        item = {'url': url, 'requested_at_utc': utcnow(), 'attempt': attempt}
        retry_delay = 2 ** attempt
        try:
            response = session.get(url, timeout=(10, 35))
            item.update(status_code=response.status_code,
                        throttle=response.headers.get('X-Throttling-Control'))
            try:
                item['response'] = response.json()
            except ValueError:
                item['response_text'] = response.text
                raise ValueError('Response was not JSON')
            payload = item['response']
            not_found = (response.status_code == 404 and
                         payload.get('Fault', {}).get('Code') == 'PUGREST.NotFound')
            if response.status_code == 200 or not_found:
                history.append(item)
                return {'status': 'not_found' if not_found else 'ok',
                        'response': payload, 'requests': history}
            item['error'] = f'HTTP {response.status_code}'
            if response.status_code not in (429, 500, 502, 503, 504):
                history.append(item)
                break
            retry_after = response.headers.get('Retry-After', '')
            if retry_after.isdigit():
                retry_delay = max(retry_delay, int(retry_after))
        except (requests.RequestException, ValueError) as exc:
            item['error'] = str(exc)
        history.append(item)
        if attempt < MAX_ATTEMPTS:
            time.sleep(retry_delay)
    return {'status': 'error', 'requests': history,
            'error': history[-1].get('error', 'Unknown error')}

def search_pubchem(term):
    result = request_json(f'{BASE_URL}/compound/name/{quote(term, safe="")}/cids/JSON')
    if result['status'] == 'ok':
        ids = result['response'].get('IdentifierList', {}).get('CID')
        if not isinstance(ids, list) or not ids or not all(isinstance(cid, int) for cid in ids):
            result.update(status='error', error='Missing or invalid candidate CID list')
        else:
            result['cids'] = sorted(set(ids))
    result.setdefault('cids', [])
    return result


## 3. Search all inventory terms in memory
Rerunning this cell reuses completed in-memory results and retries errors. It does not substitute fuzzy guesses or select a single CID from an ambiguous response.


In [4]:
search_results = globals().get('search_results', {})
collection_started_at = globals().get('collection_started_at', utcnow())
for i, (key, group) in enumerate(query_inventory.items(), 1):
    if search_results.get(key, {}).get('status') not in ('ok', 'not_found'):
        search_results[key] = search_pubchem(group['query'])
    if i % 25 == 0 or i == len(query_inventory):
        counts = Counter(r['status'] for r in search_results.values())
        print(f'Searched {i}/{len(query_inventory)}: {dict(counts)}', flush=True)

candidate_cids = sorted({cid for key in query_inventory
                         for cid in search_results[key]['cids']})
print('Unique candidate CIDs:', len(candidate_cids))


Searched 25/739: {'ok': 7, 'not_found': 18}
Searched 50/739: {'ok': 13, 'not_found': 37}
Searched 75/739: {'ok': 19, 'not_found': 56}
Searched 100/739: {'ok': 23, 'not_found': 77}
Searched 125/739: {'ok': 36, 'not_found': 89}
Searched 150/739: {'ok': 43, 'not_found': 107}
Searched 175/739: {'ok': 55, 'not_found': 120}
Searched 200/739: {'ok': 64, 'not_found': 136}
Searched 225/739: {'ok': 77, 'not_found': 148}
Searched 250/739: {'ok': 90, 'not_found': 160}
Searched 275/739: {'ok': 100, 'not_found': 175}
Searched 300/739: {'ok': 108, 'not_found': 192}
Searched 325/739: {'ok': 114, 'not_found': 211}
Searched 350/739: {'ok': 129, 'not_found': 221}
Searched 375/739: {'ok': 131, 'not_found': 244}
Searched 400/739: {'ok': 131, 'not_found': 269}
Searched 425/739: {'ok': 138, 'not_found': 287}
Searched 450/739: {'ok': 151, 'not_found': 299}
Searched 475/739: {'ok': 158, 'not_found': 317}
Searched 500/739: {'ok': 163, 'not_found': 337}
Searched 525/739: {'ok': 166, 'not_found': 359}
Searched 55

## 4. Retrieve unique compounds in batches
Properties include title, formula, weight, structure identifiers and InChIKey. Synonyms remain unfiltered raw data. Batches reduce duplicate HTTP calls. Failed batches fall back to individual requests so one missing record does not hide other records. Partial records remain visible in the preview.

Existing two-compound files are not overwritten. This run obtains fresh, traceable responses for its candidate set.


In [5]:
PROPERTY_NAMES = 'Title,IUPACName,MolecularFormula,MolecularWeight,ConnectivitySMILES,InChI,InChIKey'
compound_responses = globals().get('compound_responses', [])
records = globals().get('records', {})

def fetch_batch(cids, kind):
    suffix = f'property/{PROPERTY_NAMES}/JSON' if kind == 'properties' else 'synonyms/JSON'
    result = request_json(f'{BASE_URL}/compound/cid/{",".join(map(str, cids))}/{suffix}')
    result.update(kind=kind, requested_cids=cids)
    compound_responses.append(result)
    items = []
    if result['status'] == 'ok':
        payload = result['response']
        items = (payload.get('PropertyTable', {}).get('Properties', []) if kind == 'properties'
                 else payload.get('InformationList', {}).get('Information', []))
    found = set()
    for item in items:
        cid = item.get('CID')
        if cid in cids:
            records.setdefault(cid, {'cid': cid})[kind] = item
            found.add(cid)
    if len(cids) > 1:
        for cid in set(cids) - found:
            fetch_batch([cid], kind)

for kind in ('properties', 'synonyms'):
    pending = [cid for cid in candidate_cids if kind not in records.get(cid, {})]
    for offset in range(0, len(pending), BATCH_SIZE):
        fetch_batch(pending[offset:offset + BATCH_SIZE], kind)
        print(f'{kind}: {min(offset + BATCH_SIZE, len(pending))}/{len(pending)} processed', flush=True)


properties: 25/239 processed
properties: 50/239 processed
properties: 75/239 processed
properties: 100/239 processed
properties: 125/239 processed
properties: 150/239 processed
properties: 175/239 processed
properties: 200/239 processed
properties: 225/239 processed
properties: 239/239 processed
synonyms: 25/239 processed
synonyms: 50/239 processed
synonyms: 75/239 processed
synonyms: 100/239 processed
synonyms: 125/239 processed
synonyms: 150/239 processed
synonyms: 175/239 processed
synonyms: 200/239 processed
synonyms: 225/239 processed
synonyms: 239/239 processed


## 5. Inspect collection size and coverage BEFORE saving
“Matched” means PubChem returned candidates, not that identity was verified. Per-category counts overlap because shared terms are queried once. No-match cases include mixtures, extracts, organisms and other terms unsuitable for compound-name matching; they are not automatically collection failures.


In [6]:
def make_preview():
    categories = {}
    for category in SUPPLEMENTS:
        keys = [k for k, g in query_inventory.items()
                if any(category in o['search_categories'] for o in g['occurrences'])]
        categories[category] = {
            'queries': len(keys),
            'with_candidates': sum(search_results[k]['status'] == 'ok' for k in keys),
            'no_match': sum(search_results[k]['status'] == 'not_found' for k in keys),
            'errors': sum(search_results[k]['status'] == 'error' for k in keys),
            'unique_candidate_cids': len({c for k in keys for c in search_results[k]['cids']})}
    complete = [cid for cid in candidate_cids
                if all(k in records.get(cid, {}) for k in ('properties', 'synonyms'))]
    report = {'source_labels': len(dsld_provenance['label_sha256']),
              'query_groups': len(query_inventory),
              'search_status': dict(Counter(search_results[k]['status'] for k in query_inventory)),
              'ambiguous_queries': sum(len(search_results[k]['cids']) > 1 for k in query_inventory),
              'unique_candidate_cids': len(candidate_cids), 'complete_compounds': len(complete),
              'incomplete_cids': sorted(set(candidate_cids) - set(complete)),
              'synonym_entries': sum(len(records.get(cid, {}).get('synonyms', {}).get('Synonym', []))
                                     for cid in candidate_cids),
              'missing_property_fields': {field: sum(not records.get(cid, {}).get('properties', {}).get(field)
                                                     for cid in complete)
                                          for field in PROPERTY_NAMES.split(',')},
              'by_category': categories}
    bundle = {'source': 'PubChem PUG REST', 'started_at_utc': collection_started_at,
              'finished_at_utc': utcnow(), 'scope': 'DSLD active ingredient rows, nested rows and forms; includes co-ingredients',
              'dsld_provenance': dsld_provenance, 'queries': query_inventory,
              'search_results': {k: search_results[k] for k in query_inventory},
              'compound_responses': compound_responses,
              'records': {str(cid): records[cid] for cid in candidate_cids if cid in records}}
    report['payload_bytes_before_report'] = len(json.dumps(bundle, ensure_ascii=False).encode('utf-8'))
    bundle['report'] = report
    return report, bundle

preview_report, collection_bundle = make_preview()
print(json.dumps(preview_report, indent=2))
print('Approximate payload MiB:', round(preview_report['payload_bytes_before_report'] / 1024**2, 3))
print('No new PubChem data has been saved.')


{
  "source_labels": 400,
  "query_groups": 739,
  "search_status": {
    "ok": 228,
    "not_found": 511
  },
  "ambiguous_queries": 7,
  "unique_candidate_cids": 239,
  "complete_compounds": 239,
  "incomplete_cids": [],
  "synonym_entries": 46426,
  "missing_property_fields": {
    "Title": 0,
    "IUPACName": 3,
    "MolecularFormula": 0,
    "MolecularWeight": 0,
    "ConnectivitySMILES": 0,
    "InChI": 0,
    "InChIKey": 0
  },
  "by_category": {
    "Magnesium": {
      "queries": 52,
      "with_candidates": 30,
      "no_match": 22,
      "errors": 0,
      "unique_candidate_cids": 32
    },
    "Vitamin D": {
      "queries": 58,
      "with_candidates": 41,
      "no_match": 17,
      "errors": 0,
      "unique_candidate_cids": 42
    },
    "Ashwagandha": {
      "queries": 40,
      "with_candidates": 9,
      "no_match": 31,
      "errors": 0,
      "unique_candidate_cids": 9
    },
    "Melatonin": {
      "queries": 12,
      "with_candidates": 3,
      "no_match": 9,


## 6. Optional save after inspecting the preview
Set `SAVE_DATA = True` here only after reviewing the preceding report. Save an immutable run directory containing raw HTTP responses, query provenance, candidate records and a manifest. Previous collection files are retained. Incomplete runs are explicitly marked; failed requests are available for diagnosis.


In [ ]:
# SAVE_DATA = True
if SAVE_DATA:
    run_dir = RAW_DIR / 'runs' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    run_dir.mkdir(parents=True, exist_ok=False)
    payload = json.dumps(collection_bundle, ensure_ascii=False, indent=2).encode('utf-8')
    (run_dir / 'collection.json').write_bytes(payload)
    manifest = {'source': 'PubChem', 'saved_at_utc': utcnow(),
                'collection_file': 'collection.json', 'bytes': len(payload),
                'sha256': hashlib.sha256(payload).hexdigest(),
                'complete': not preview_report['incomplete_cids'] and not preview_report['search_status'].get('error', 0),
                'report': preview_report}
    (run_dir / 'collection_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print('Saved:', run_dir)
else:
    print('Preview only. Data remains in memory; SAVE_DATA is False.')


## Executed collection result

The preview was inspected before saving this run to `data/raw/pubchem/runs/20261028T035149634190Z`. The saved manifest records 9,442,748 bytes and a SHA-256 checksum. All 239 candidate compounds have property and synonym responses; 3 lack IUPACName. There were 228 queries with candidates, 511 no-match queries and no request errors. Seven queries returned multiple candidates. Candidate identities have not been manually verified.

`SAVE_DATA` remains `False` by default for future runs.
